# Seminar 7: Récursion

**Université de Genève — Algorithmique et structures de données**

---

> **Book reference:** Week 7 follows the **Récursion** chapter ([*Problem Solving with Algorithms and Data Structures using Python*](https://runestone.academy/ns/books/published/pythonds/Recursion/toctree.html)), numbered **Chapter 4** in the Runestone edition. La lecture de ce chapitre en parallèle du séminaire est fortement recommandée.

## Objectifs d'apprentissage

À la fin de ce séminaire, vous serez capable de :

- Énoncer les trois lois de la récursion et les appliquer à de nouveaux problèmes
- Suivre les appels récursifs à l'aide d'un diagramme de pile d'appels
- Implémenter factorielle, conversion de base et Tour de Hanoï de façon récursive
- Expliquer pourquoi la version naïve de Fibonacci est en O(2ⁿ) et comment la mémoïsation la ramène à O(n)
- Savoir quand la récursion est l'outil approprié — et quand préférer l'itération

---

## Part 1: Theory

### 1.1 Qu'est-ce que la récursion ?

Une fonction est **récursive** lorsqu'elle s'appelle elle‑même pour résoudre une version plus petite du même problème. Chaque appel récursif réduit le problème jusqu'à atteindre un **cas de base** — une version suffisamment simple pour être résolue directement sans récursion supplémentaire.

#### Les trois lois de la récursion

1. **Un algorithme récursif doit avoir un cas de base.** Le cas de base arrête la récursion. Sans lui, les appels continuent indéfiniment et Python lève `RecursionError`.
2. **Un algorithme récursif doit modifier son état et se diriger vers le cas de base.** Chaque appel doit rendre le problème strictement plus petit (moins d'éléments, nombre inférieur, chaîne plus courte).
3. **Un algorithme récursif doit s'appeler lui‑même (récursivement).** C'est ce qui le rend récursif.

#### Analogie : poupées russes emboîtées

Imaginez un ensemble de poupées russes emboîtées (matrioshkas). Chaque poupée contient une poupée plus petite, identique — jusqu'à la plus petite qui n'en contient plus (le cas de base). Ouvrir une poupée revient à faire un appel récursif ; la plus petite poupée représente le cas de base.

#### Itératif vs Récursif : somme d'une liste

Les deux approches donnent le même résultat ; la récursion exprime la même logique de façon auto‑référentielle :

```python
# Iterative
def list_sum_iter(lst):
    total = 0
    for n in lst:
        total += n
    return total

# Recursive
def list_sum(lst):
    if len(lst) == 0:        # base case: empty list sums to 0
        return 0
    return lst[0] + list_sum(lst[1:])   # recursive step
```

La version récursive se lit ainsi : *la somme d'une liste est son premier élément plus la somme du reste*.

In [ ]:
def list_sum(lst):
    """Return the sum of a list recursively."""
    if len(lst) == 0:
        return 0
    return lst[0] + list_sum(lst[1:])

# Verify against built-in
data = [1, 3, 5, 7, 9]
assert list_sum(data) == sum(data) == 25
assert list_sum([]) == 0
print("list_sum works correctly")

### 1.2 Schémas récursifs simples

#### Countdown

```python
def countdown(n):
    if n <= 0:          # base case
        print("Go!")
        return
    print(n)
    countdown(n - 1)   # recursive step: n decreases each call
```

#### Factorielle

n! = n × (n−1) × … × 1, avec 0! = 1

```python
def factorial(n):
    if n == 0:          # base case
        return 1
    return n * factorial(n - 1)   # recursive step
```

#### Inversion d'une chaîne de caractères

```python
def reverse_str(s):
    if len(s) <= 1:    # base case: empty string or single char
        return s
    return s[-1] + reverse_str(s[:-1])  # last char + reverse of the rest
```

#### Factorielle — analyse détaillée

The recursive rule is:

- `factorial(0) = 1` (base case)
- `factorial(n) = n * factorial(n-1)` for `n > 0`

For `factorial(5)`, the call phase and return phase are different:

```text
Call phase:
factorial(5) -> factorial(4) -> factorial(3) -> factorial(2) -> factorial(1) -> factorial(0)

Return phase:
factorial(0) = 1
factorial(1) = 1 * 1 = 1
factorial(2) = 2 * 1 = 2
factorial(3) = 3 * 2 = 6
factorial(4) = 4 * 6 = 24
factorial(5) = 5 * 24 = 120
```

C'est pourquoi la récursion se conçoit en deux étapes : **descendre** jusqu'au cas de base, puis **remonter** la réponse finale lors du retour.

In [ ]:
def factorial_verbose(n, depth=0):
    """Print call/return trace, then return n!."""
    indent = "  " * depth
    print(f"{indent}call factorial({n})")

    if n == 0:
        print(f"{indent}return 1  (base case)")
        return 1

    sub = factorial_verbose(n - 1, depth + 1)
    result = n * sub
    print(f"{indent}return {n} * {sub} = {result}")
    return result

print("Tracing factorial(5):")
value = factorial_verbose(5)
print("Final result:", value)

In [ ]:
def countdown(n):
    if n <= 0:
        print("Go!")
        return
    print(n)
    countdown(n - 1)

def factorial(n):
    """Return n! recursively. n must be a non-negative integer."""
    if n == 0:
        return 1
    return n * factorial(n - 1)

def reverse_str(s):
    """Return the reverse of string s recursively."""
    if len(s) <= 1:
        return s
    return s[-1] + reverse_str(s[:-1])

countdown(5)
print()
print("5! =", factorial(5))    # 120
print("10! =", factorial(10))  # 3628800
print(reverse_str("hello"))    # 'olleh'

### 1.3 Cadres de pile — comment Python exécute la récursion

Python utilise la **pile d'appels** (la même structure LIFO de la Semaine 6 !) pour exécuter les fonctions récursives. À chaque appel de fonction, Python empile un nouveau **cadre** sur la pile. Le cadre contient :

- Les variables locales pour cet appel
- Le pointeur d'instruction (où revenir)

Quand la fonction retourne, son cadre est **retiré** de la pile.

#### Trace de la pile d'appels pour `factorial(4)`

```
factorial(4)  ← pushed, waits for factorial(3)
  factorial(3)  ← pushed, waits for factorial(2)
    factorial(2)  ← pushed, waits for factorial(1)
      factorial(1)  ← pushed, waits for factorial(0)
        factorial(0)  ← BASE CASE → returns 1
      ← pops, returns 1 × 1 = 1
    ← pops, returns 2 × 1 = 2
  ← pops, returns 3 × 2 = 6
← pops, returns 4 × 6 = 24
```

Chaque niveau indenté correspond à un cadre sur la pile d'appels. La profondeur maximale de la pile à un instant donné est égale à la profondeur récursive.

#### Limite de récursion

Python impose par défaut une limite de récursion de 1000 cadres (`sys.getrecursionlimit()`). La dépasser lève `RecursionError`. C'est une fonctionnalité, pas un bug — elle empêche une récursion infinie accidentelle de faire planter votre machine.

In [ ]:
import sys

print("Default recursion limit:", sys.getrecursionlimit())

# Demonstrate the call stack depth
def depth_counter(n, current_depth=0):
    """Count recursion depth by passing it along."""
    if n == 0:
        return current_depth
    return depth_counter(n - 1, current_depth + 1)

print("Depth of factorial(10):", depth_counter(10), "frames")
print("Depth of factorial(100):", depth_counter(100), "frames")

### 1.4 Conversion de base

Convertir un entier dans une base différente (par exemple binaire ou hexadécimal) produit des restes dans l'ordre *inverse*. Une pile inverse naturellement une séquence — et la récursion est équivalente à une pile implicite.

**Algorithme :**
- Cas de base : si `n < base`, retourner directement le chiffre unique
- Étape récursive : convertir d'abord `n // base` (chiffres de poids fort), puis ajouter `n % base`

```
to_str(769, 10):
  769 // 10 = 76 → to_str(76, 10) → recurse
    76 // 10 = 7  → to_str(7, 10)  → base case → '7'
    76 %  10 = 6  → '6'
  769 % 10  = 9   → '9'
Result: '769'
```

In [ ]:
def to_str(n, base):
    """Convert non-negative integer n to a string in the given base (2-16)."""
    digits = '0123456789ABCDEF'
    if n < base:            # base case: single digit
        return digits[n]
    return to_str(n // base, base) + digits[n % base]

print(to_str(769, 10))   # '769'
print(to_str(10,  2))    # '1010'  (binary)
print(to_str(255, 16))   # 'FF'    (hex)
print(to_str(8,   2))    # '1000'

# Verify against built-in
assert to_str(255, 16) == 'FF'
assert to_str(10, 2) == bin(10)[2:].upper()
print("All assertions passed")

### 1.5 Visualiser la récursion avec Turtle (optionnel)

La récursion devient beaucoup plus facile à comprendre lorsque l'on peut **voir** la structure répétée.

Le module `turtle` de Python est utile pour cela car :
- chaque appel récursif dessine une version plus petite du même motif,
- le cas de base arrête le dessin,
- l'image finale rend visible le schéma d'appels.

Un exemple courant est un **arbre fractal** :
- dessiner une branche,
- dessiner récursivement une petite branche à gauche,
- dessiner récursivement une petite branche à droite.

> **Exam note:** Vous n'êtes **pas** censé(e) mémoriser les détails de l'API de `turtle`.  
> Pour l'évaluation, concentrez‑vous sur la logique récursive : cas de base, sous‑problème plus petit, et auto‑appels.

#### Si `turtle` ne fonctionne pas dans votre notebook

Certains kernels Jupyter n'incluent pas `tkinter`, requis par `turtle`. Si vous obtenez une erreur du type `No module named '_tkinter'`, exécutez le code comme script autonome depuis un terminal à la place :

```bash
python <your-pythonfil-ename>.py
```

In [ ]:
# Recursive turtle demo: fractal tree
# Note: this opens a GUI window in local Python environments.
import turtle


def tree(branch_len, t):
    if branch_len <= 5:      # base case
        return

    t.forward(branch_len)

    t.right(20)
    tree(branch_len - 15, t)  # recursive call 1

    t.left(40)
    tree(branch_len - 15, t)  # recursive call 2

    t.right(20)
    t.backward(branch_len)


screen = turtle.Screen()
screen.title("Recursive Fractal Tree")

pen = turtle.Turtle()
pen.left(90)
pen.speed("fastest")
pen.color("black")

tree(75, pen)
screen.exitonclick()

### 1.6 Tour de Hanoï (Livre §4.5)

**Problème :** Déplacer `n` disques de la tige A vers la tige C, en utilisant la tige B comme auxiliaire.

**Règles :**
- Un seul disque peut être déplacé à la fois.
- Un disque plus grand ne peut jamais être placé sur un disque plus petit.

**Pourquoi l'itération est difficile :** Il faut suivre l'état de trois tiges en même temps. La logique pour n tiges implique un nombre exponentiel de cas.

**Intuition récursive :** Pour déplacer `n` disques de A vers C :
1. Déplacer les `n−1` disques du dessus de A vers B (récursion, en utilisant C comme auxiliaire)
2. Déplacer le disque `n` de A vers C (étape élémentaire)
3. Déplacer les `n−1` disques de B vers C (récursion, en utilisant A comme auxiliaire)

Cela décompose le problème en deux sous‑problèmes identiques de taille `n−1`.

**Nombre de déplacements :** 2ⁿ − 1 déplacements sont nécessaires pour n disques. C'est démontrablement optimal.

In [ ]:
def move_tower(n, from_peg, to_peg, aux_peg):
    """Print the moves to transfer n disks from from_peg to to_peg."""
    if n == 1:   # base case: move a single disk
        print(f"Move disk 1 from {from_peg} to {to_peg}")
        return
    move_tower(n - 1, from_peg, aux_peg, to_peg)    # step 1
    print(f"Move disk {n} from {from_peg} to {to_peg}")    # step 2
    move_tower(n - 1, aux_peg, to_peg, from_peg)    # step 3

print("Tower of Hanoi — n=3:")
move_tower(3, 'A', 'C', 'B')
print("\n(should be 2^3 - 1 = 7 moves)")

### 1.7 Fibonacci : naïf vs mémoïsation

The Fibonacci sequence is defined as:

```
fib(0) = 0
fib(1) = 1
fib(n) = fib(n-1) + fib(n-2)  for n ≥ 2
```

#### Le piège exponentiel

La récursion naïve recalcule les mêmes sous‑problèmes à plusieurs reprises :

```
fib(5)
├── fib(4)
│   ├── fib(3)         ← computed twice!
│   │   ├── fib(2)
│   │   └── fib(1)
│   └── fib(2)
└── fib(3)             ← computed twice!
    ├── fib(2)
    └── fib(1)
```

Cela conduit à **O(2ⁿ)** appels totaux — `fib(40)` requiert plus d'un milliard d'appels récursifs.

#### Mémoïsation : mettre en cache ce que vous avez déjà calculé

Stockez les résultats dans un dictionnaire. Avant de calculer, vérifiez si le résultat est déjà en cache.
Cela réduit la complexité de **O(2ⁿ) → O(n)**.

> **Aperçu :** La mémoïsation est l'idée centrale derrière la **programmation dynamique**, que vous étudierez en Semaine 11.

In [ ]:
# Naive: O(2^n) — definition only (counting calls is Exercise 2.4)
def fib_naive_demo(n: int) -> int:
    if n <= 1:
        return n
    return fib_naive_demo(n - 1) + fib_naive_demo(n - 2)


# Memoised: O(n)
def fib_memo(n, cache=None):
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    if n <= 1:
        return n
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]


print("Fibonacci values (both methods should agree):")
for n in [0, 1, 5, 10, 20]:
    assert fib_memo(n) == fib_naive_demo(n), f"Mismatch at n={n}"
    print(f"  fib({n}) = {fib_memo(n)}")
print("All values match!")

---

## Partie 2 : Exercices

Travaillez les exercices **2.1–2.4** dans l'ordre. Utilisez Python pur. Dans l'Exercice 2.1, n'appelez **pas** `math.factorial` dans vos propres implémentations — vous pouvez utiliser `math.factorial` uniquement dans la cellule de test séparée pour vérifier les résultats. Évitez `functools.reduce` sauf si un exercice l'indique. Ajoutez vos propres vérifications `assert` lorsque cela est utile.

### Exercice 2.1 — Factorielle : récursive et itérative

**Partie A.** Implémentez `factorial_recursive(n)` et `factorial_iterative(n)` à partir de zéro (n'appelez pas `math.factorial`).

**Partie B.** Pour les deux implémentations, mesurez la profondeur d'appel pour n = 10, 100, 500. À quelle valeur de n la version récursive plante‑t‑elle ? (Indice : utilisez `sys.getrecursionlimit()`.)

**Partie C.** Quelle version préférez‑vous pour les grandes valeurs de n ? Pourquoi ?

In [ ]:
# Exercise 2.1 — implementations (run the next cell to test when ready)

def factorial_recursive(n: int) -> int:
    """Return n! using recursion. Raises ValueError for negative n."""
    # Hint: base case n == 0 → 1; else n * factorial_recursive(n - 1)
    raise NotImplementedError("Replace this with your recursive factorial.")


def factorial_iterative(n: int) -> int:
    """Return n! using iteration. Raises ValueError for negative n."""
    # Hint: start with result = 1, loop k from 2 to n (or 1 to n), result *= k
    raise NotImplementedError("Replace this with your iterative factorial.")


In [ ]:
# Exercise 2.1 — tests (OK to use math.factorial here only, for checking)
import math

for k in [0, 1, 5, 10]:
    assert factorial_recursive(k) == math.factorial(k), f"Recursive wrong at {k}"
    assert factorial_iterative(k) == math.factorial(k), f"Iterative wrong at {k}"
print("Basic tests passed")

# Part B — experiment with recursion depth vs sys.getrecursionlimit() here


### Exercice 2.2 — Inverser une chaîne récursivement

Implémentez `reverse_string(s: str) -> str` **récursivement** (pas de `s[::-1]`, pas de `reversed()`).

- Cas de base : une chaîne de longueur 0 ou 1 est déjà inversée
- Étape récursive : la chaîne inversée est le dernier caractère suivi de l'inversion de tout ce qui le précède

In [ ]:
# Exercise 2.2 — implementation (run the next cell to test)

def reverse_string(s: str) -> str:
    """Return the reverse of s using recursion."""
    # Hint: base case len(s) <= 1; else s[-1] + reverse_string(s[:-1])
    raise NotImplementedError("Replace this with your recursive reverse.")

In [ ]:
# Exercise 2.2 — tests
assert reverse_string("") == ""
assert reverse_string("a") == "a"
assert reverse_string("hello") == "olleh"
assert reverse_string("racecar") == "racecar"
print("All tests passed")

### Exercice 2.3 — Tour de Hanoï

**Partie A.** Implémentez `move_tower(n, from_peg, to_peg, aux_peg)` qui **retourne une liste de chaînes décrivant les mouvements** (au lieu de les afficher), par exemple `['Move disk 1 from A to C', ...]`.

**Partie B.** Appelez votre fonction pour n = 4 et affichez la séquence de mouvements. Combien de mouvements faut‑il ?

**Partie C.** Vérifiez que le nombre de mouvements est égal à 2ⁿ − 1 pour n = 1, 2, 3, 4, 5 en utilisant un `assert`.

In [ ]:
# Exercise 2.3 — implementation
# Same three steps as the printed move_tower in §1.6, but collect strings in lists:
#   n==1 → return [f"Move disk 1 from {from_peg} to {to_peg}"]
#   else → moves_small + [f"Move disk {n} ..."] + moves_small (with pegs permuted)

def move_tower(n, from_peg, to_peg, aux_peg) -> list:
    """Return the list of moves to transfer n disks from from_peg to to_peg."""
    raise NotImplementedError("Replace this with your list-returning move_tower.")

In [ ]:
# Exercise 2.3 — Parts B & C (run after your move_tower is implemented)

moves_4 = move_tower(4, 'A', 'C', 'B')
print(f"n=4: {len(moves_4)} moves")
for m in moves_4:
    print(" ", m)

for n in [1, 2, 3, 4, 5]:
    moves = move_tower(n, 'A', 'C', 'B')
    assert len(moves) == 2**n - 1, f"Expected {2**n - 1} moves for n={n}, got {len(moves)}"
print("Move count formula 2^n - 1 verified for n=1..5")

### Exercice 2.4 — Fibonacci : naïf puis mémoïsation

**Partie A.** Implémentez `fib_naive(n)` en utilisant la récursion simple (sans cache).

**Partie B.** Implémentez `fib_fast(n)` en utilisant un dictionnaire de mémoïsation.

**Partie C.** Comptez le nombre d'appels récursifs effectués par chaque version pour n = 5, 10, 15, 20, 25. Présentez les résultats dans un tableau et commentez la différence.

**Partie D.** *(Optionnel)* Tracez le nombre d'appels pour n = 5 à 35 en utilisant `matplotlib`.

In [ ]:
# Exercise 2.4 — implementations (run the next cell to test)

def fib_naive(n: int) -> int:
    """Return fib(n) using naive recursion. O(2^n) time."""
    # Hint: base n <= 1 return n; else fib_naive(n-1) + fib_naive(n-2)
    raise NotImplementedError("Replace this with naive Fibonacci.")


def fib_fast(n: int, cache: dict | None = None) -> int:
    """Return fib(n) using memoisation. O(n) time."""
    # Hint: if cache is None: cache = {}; if n in cache: return cache[n]; ...
    raise NotImplementedError("Replace this with memoised Fibonacci.")

In [ ]:
# Exercise 2.4 — correctness, then Part C (call counts)

expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
for i, val in enumerate(expected):
    assert fib_naive(i) == val, f"fib_naive({i}) wrong"
    assert fib_fast(i) == val, f"fib_fast({i}) wrong"
print("Correctness OK — now measure call counts (Part C):")

# Part C — count fib_naive body entries with a wrapper (example pattern):
#   calls = 0
#   def fib_naive_counted(n):
#       nonlocal calls
#       calls += 1
#       ... same logic as fib_naive ...
#   Then reset calls = 0 before each n and print calls for n in [5, 10, 15, 20, 25].
# For fib_fast, counting recursive entries is optional; comment on O(n) vs exponential instead.
